<a href="https://colab.research.google.com/github/2212517-PhanLeMinhPhu/test-tool-web-2/blob/main/webtool_quantracthucdia.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import streamlit as st
import pandas as pd
import json
import plotly.express as px

# Cấu hình giao diện Streamlit
st.set_page_config(page_title="Phân tích lịch tưới cây", page_icon="🌱", layout="wide")
st.title("🌱 Phân tích Lịch Tưới Cây từ Dữ Liệu Cảm Biến")

# 1. Đọc file JSON
@st.cache_data
def load_data(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    df = pd.DataFrame(data)

    # Xử lý cột thời gian
    df['Thời gian'] = pd.to_datetime(df['Thời gian'], format='%Y-%m-%d %H-%M-%S')

    # Xử lý cột 'Độ ẩm' (chuyển sang dạng số)
    if 'Độ ẩm' in df.columns:
        df['Độ ẩm'] = pd.to_numeric(df['Độ ẩm'], errors='coerce')

    # Sắp xếp theo thời gian
    df = df.sort_values(by='Thời gian').reset_index(drop=True)
    return df

try:
    df = load_data("Quan trắc thực địa (1).json")

    # 2. Lọc khung giờ (5:00 AM - 23:59 PM)
    df['Giờ'] = df['Thời gian'].dt.hour
    df_filtered = df[(df['Giờ'] >= 5) & (df['Giờ'] <= 23)].copy()

    # Lọc ra nhóm dữ liệu chứa Độ ẩm (nhóm STT 1, 2, 3 trong file của bạn đều có độ ẩm)
    df_moisture = df_filtered.dropna(subset=['Độ ẩm']).copy()
    # Loại bỏ các giá trị Độ ẩm = 0 (do lỗi cảm biến ngắt kết nối hoặc nhiễu)
    df_moisture = df_moisture[df_moisture['Độ ẩm'] > 0]

    # 3. Thuật toán phát hiện cử tưới (Tìm độ dốc gia tăng lớn)
    # Tính chênh lệch độ ẩm giữa lần đo hiện tại và lần trước đó
    df_moisture['Độ ẩm thay đổi'] = df_moisture['Độ ẩm'].diff()

    # Giả sử độ ẩm tăng đột ngột trên 15 đơn vị thì được tính là 1 lần bắt đầu tưới
    nguong_tuoi = st.sidebar.number_input("Ngưỡng độ ẩm tăng (để tính là 1 lần tưới):", min_value=5, max_value=100, value=15)

    # Lọc ra các điểm tưới
    df_tuoi = df_moisture[df_moisture['Độ ẩm thay đổi'] >= nguong_tuoi]

    # 4. Hiển thị thông tin tổng quan
    st.header(f"💦 Tổng số cử tưới trong ngày: {len(df_tuoi)} lần")
    st.markdown("**(Khung giờ đã lọc: 05:00 đến 23:59)**")

    if len(df_tuoi) > 0:
        with st.expander("Xem chi tiết thời gian tưới"):
            st.dataframe(df_tuoi[['Thời gian', 'Độ ẩm', 'Độ ẩm thay đổi']].style.format({'Thời gian': lambda t: t.strftime("%H:%M:%S (%d/%m/%Y)")}))

    # 5. Vẽ biểu đồ dạng dây (Line Chart)
    st.subheader("📈 Biểu đồ biến thiên Độ ẩm")

    fig = px.line(df_moisture, x='Thời gian', y='Độ ẩm',
                  title='Biến thiên Độ ẩm đất (từ 5h - 23h59)',
                  labels={'Thời gian': 'Thời Gian', 'Độ ẩm': 'Độ ẩm (Giá trị cảm biến)'},
                  markers=True)

    # Đánh dấu các điểm được cho là bắt đầu cử tưới trên biểu đồ
    fig.add_scatter(x=df_tuoi['Thời gian'], y=df_tuoi['Độ ẩm'],
                    mode='markers', marker=dict(color='red', size=12, symbol='star'),
                    name='Điểm bắt đầu tưới')

    fig.update_traces(line_color='#2e7d32') # Màu xanh lá
    st.plotly_chart(fig, use_container_width=True)

except FileNotFoundError:
    st.error("Không tìm thấy file 'Quan trắc thực địa (1).json'. Hãy đảm bảo file này nằm cùng thư mục với đoạn code.")